# [HumanEvalPack](https://huggingface.co/datasets/bigcode/humanevalpack)
HumanEvalPack is an extension of OpenAI's HumanEval to cover 6 total languages across 3 tasks. The Python split is exactly the same as OpenAI's Python HumanEval. The other splits are translated by humans (similar to HumanEval-X but with additional cleaning, see [here](https://github.com/bigcode-project/octopack/tree/main/evaluation/create/humaneval-x#modifications-muennighoff)). Refer to the OctoPack paper for more details.

Languages: Python, JavaScript, Java, Go, C++, Rust

The data fields are the same among all splits:

- `task_id`: Indicates the language (Python/JavaScript/Java/Go/C++/Rust) and task id (from 0 to 163) of the problem  
- `prompt`: the prompt for models relying on code continuation  
- `declaration`: the declaration of the function (same as prompt but without the docstring)  
- `canonical_solution`: the correct solution passing all unit tests for the problem  
- `buggy_solution`: same as canonical_solution but with a subtle human-written bug causing the unit tests to fail  
- `bug_type`: the type of the bug in buggy_solution (one of [missing logic, excess logic, value misuse, operator misuse, variable misuse, function misuse])  
- `failure_symptoms`: the problem the bug causes (one of [incorrect output, stackoverflow, infinite loop])  
- `entry_point`: the name of the function  
- `import`: imports necessary for the solution (only present for Go)  
- `test_setup`: imports necessary for the test execution (only present for Go)  
- `test`: the unit tests for the problem  
- `example_test`: additional unit tests different from test that could be e.g. provided to the model (these are not used in the paper)  
- `signature`: the signature of the function  
- `docstring`: the docstring describing the problem  
- `instruction`: an instruction for HumanEvalSynthesize in the form Write a {language_name} function {signature} to solve the following problem:\n{docstring}

In [11]:
# pip install -q datasets
from datasets import load_dataset
# Languages: "python", "js", "java", "go", "cpp", "rust"
ds = load_dataset("bigcode/humanevalpack", "cpp")
ds


/home/gkoren/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dict_keys(['test'])

In [3]:
print(f"there are {ds['test'].num_rows} samples in the dataset")
print("each sample has the following keys: (see explanation above)")
print(ds['test'][0].keys())

there are 164 samples in the dataset
each sample has the following keys: (see explanation above)
dict_keys(['task_id', 'prompt', 'declaration', 'canonical_solution', 'buggy_solution', 'bug_type', 'failure_symptoms', 'entry_point', 'import', 'test_setup', 'test', 'example_test', 'signature', 'docstring', 'instruction'])


In [12]:
# looking at some keys:
def print_sample(eidx):
    print('='*20,f"problem {eidx}",'='*20)
    print('-'*10,f"prompt",'-'*10)
    print(ds['test'][eidx]['prompt'])
    # print(ds['test'][eidx]['declaration'])
    print('-'*10,f"test",'-'*10)
    print(ds['test'][eidx]['test'])
    print('-'*10,f"solution",'-'*10)
    print(ds['test'][eidx]['canonical_solution'])

eidx=10
print_sample(eidx)

==================== problem 10 ====================
---------- prompt ----------
#include<stdio.h>
#include<string>
using namespace std;
bool is_palindrome(string str){
    //Test if given string is a palindrome 
    string s(str.rbegin(),str.rend());
    return s==str;
}
string make_palindrome(string str){
    /*
    Find the shortest palindrome that begins with a supplied string. 
    Algorithm idea is simple: - Find the longest postfix of supplied string that is a palindrome. 
    - Append to the end of the string reverse of a string prefix that comes before the palindromic suffix.
    >>> make_palindrome("") 
    "" 
    >>> make_palindrome("cat") 
    "catac" 
    >>> make_palindrome("cata") 
    "catac" 
    */

---------- test ----------
#undef NDEBUG
#include<assert.h>
int main(){
    assert (make_palindrome("") == "");
    assert (make_palindrome("x") == "x");
     assert (make_palindrome("xyz") == "xyzyx");
     assert (make_palindrome("xyx") == "xyx") ;
     assert (make_pa

In [15]:
# explore the format of generation file
import json

generations_file='../outputs/generations_codeqwen15_humanevalsynthesize-cpp.json'
# Open and read the JSON file
with open(generations_file, 'r') as file:
    generations = json.load(file)

print(f'found {len(generations)} problems, each has {len(generations[0])} generated solutions')

found 164 problems, each has 20 generated solutions


In [19]:
eidx=16
print_sample(eidx)
print('='*30,'generated code','='*30)
print(generations[eidx][0])

==================== problem 16 ====================
---------- prompt ----------
/*
Given a string, find out how many distinct characters (regardless of case) does it consist of
>>> count_distinct_characters("xyzXYZ")
3
>>> count_distinct_characters("Jerry")
4
*/
#include<stdio.h>
#include<vector>
#include<string>
#include<algorithm>
using namespace std;
int count_distinct_characters(string str){ 

---------- test ----------
#undef NDEBUG
#include<assert.h>
int main(){
    assert (count_distinct_characters("") == 0);
    assert (count_distinct_characters("abcde") == 5);
    assert (count_distinct_characters("abcdecadeCADE") == 5);
    assert (count_distinct_characters("aaaaAAAAaaaa") == 1);
    assert (count_distinct_characters("Jerry jERRY JeRRRY") == 5);
}

---------- solution ----------
    vector<char> distinct={};
    transform(str.begin(),str.end(),str.begin(),::tolower);
    for (int i=0;i<str.size();i++)
    {
        bool isin=false;
        for (int j=0;j<distinct.size();j++

In [18]:
eidx=1
all(x==generations[eidx][0] for x in generations[eidx])

False